# Rainfall Analysis and Planting Advisory

**Scenario:** A farmer-support NGO sends SMS planting advice to farmers in five counties. This notebook cleans the Machakos station data (typed in by hand), analyses monthly rainfall with NumPy, and produces a short planting advisory.

**Counties:** Nairobi, Nakuru, Kisumu, Mombasa, Machakos  |  **Unit:** millimetres (mm)

## 1. Data and imports

In [1]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## 2. Clean the Machakos data

`int()` raises a `ValueError` when a string cannot be converted (for example `"N/A"` or an empty string). The function catches **only** `ValueError`, replaces invalid entries with the rounded average of the valid Machakos values, and reports which months were imputed. Note that `" 20 "` is valid: `int()` ignores surrounding spaces.

In [2]:
def clean_machakos(raw_values):
    """Convert raw strings to ints. Invalid entries are replaced by the
    rounded mean of the valid values. Returns (cleaned, imputed_indices, fill)."""
    cleaned = []
    imputed_idx = []

    for i, value in enumerate(raw_values):
        try:
            cleaned.append(int(value))
        except ValueError:
            cleaned.append(None)      # placeholder until the mean is known
            imputed_idx.append(i)

    valid = [v for v in cleaned if v is not None]
    fill = round(sum(valid) / len(valid))

    for i in imputed_idx:
        cleaned[i] = fill

    return cleaned, imputed_idx, fill


machakos_clean, imputed_idx, fill_value = clean_machakos(machakos_raw)

print("Imputed months:", [months[i] for i in imputed_idx])
print("Imputed value (mm):", fill_value)
print("Cleaned Machakos row:", machakos_clean)

Imputed months: ['Apr', 'Sep']
Imputed value (mm): 68
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## 3. Combine into a single 5 × 12 NumPy array

In [3]:
R = np.array(clean_rows + [machakos_clean])

print("Shape:", R.shape)
print("Dtype:", R.dtype)
print(R)

Shape: (5, 12)
Dtype: int64
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## 4. Annual totals and monthly averages

`axis=1` collapses the months (one total per county). `axis=0` collapses the counties (one average per month).

In [4]:
annual_totals = R.sum(axis=1)
monthly_avg = R.mean(axis=0)

print("Annual total rainfall (mm) per county:")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<9} {total}")

print("\nAverage rainfall per month across all counties (mm):")
for month, avg in zip(months, monthly_avg):
    print(f"  {month}  {avg:.1f}")

Annual total rainfall (mm) per county:
  Nairobi   1045
  Nakuru    916
  Kisumu    1371
  Mombasa   1075
  Machakos  811

Average rainfall per month across all counties (mm):
  Jan  47.0
  Feb  47.6
  Mar  91.0
  Apr  164.4
  May  171.2
  Jun  67.6
  Jul  56.6
  Aug  58.2
  Sep  64.2
  Oct  67.8
  Nov  127.4
  Dec  80.6


## 5. Wettest month per county

`np.argmax(..., axis=1)` returns the column index of the maximum in each row; `zip()` pairs each county with its result.

In [5]:
wettest_idx = np.argmax(R, axis=1)
wettest_month = np.array(months)[wettest_idx]

for county, month, idx in zip(counties, wettest_month, wettest_idx):
    print(f"{county:<9} wettest month: {month} ({R[counties.index(county), idx]} mm)")

Nairobi   wettest month: Apr (219 mm)
Nakuru    wettest month: Apr (145 mm)
Kisumu    wettest month: Apr (240 mm)
Mombasa   wettest month: May (240 mm)
Machakos  wettest month: Nov (182 mm)


## 6. Dry months (below 50 mm)

A boolean mask marks each dry month as `True`; summing along `axis=1` counts the `True` values per county.

In [6]:
dry_mask = R < 50
dry_counts = dry_mask.sum(axis=1)

for county, n in zip(counties, dry_counts):
    print(f"{county:<9} dry months: {n}")

Nairobi   dry months: 4
Nakuru    dry months: 3
Kisumu    dry months: 0
Mombasa   dry months: 2
Machakos  dry months: 6


## 7. Recommended planting window (two consecutive months)

`R[:, :-1] + R[:, 1:]` adds each month to the next one, giving 11 two-month totals per county. `argmax` then picks the best pair.

In [7]:
pair_totals = R[:, :-1] + R[:, 1:]            # shape (5, 11)
start_idx = np.argmax(pair_totals, axis=1)    # first month of best pair
best_totals = pair_totals.max(axis=1)

month_arr = np.array(months)
planting_window = np.char.add(np.char.add(month_arr[start_idx], "-"), month_arr[start_idx + 1])

for county, window, total in zip(counties, planting_window, best_totals):
    print(f"{county:<9} planting window: {window} ({total} mm combined)")

Nairobi   planting window: Apr-May (395 mm combined)
Nakuru    planting window: Apr-May (265 mm combined)
Kisumu    planting window: Apr-May (420 mm combined)
Mombasa   planting window: Apr-May (390 mm combined)
Machakos  planting window: Nov-Dec (278 mm combined)


## 8. Write `rainfall_summary.csv`

Written with the `csv` module: one header row and five county rows.

In [8]:
header = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for row in zip(counties, annual_totals.tolist(), wettest_month.tolist(),
                   dry_counts.tolist(), planting_window.tolist()):
        writer.writerow(row)

# Read the file back to confirm it is correct
with open("rainfall_summary.csv", newline="") as f:
    for line in csv.reader(f):
        print(line)

['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
['Nairobi', '1045', 'Apr', '4', 'Apr-May']
['Nakuru', '916', 'Apr', '3', 'Apr-May']
['Kisumu', '1371', 'Apr', '0', 'Apr-May']
['Mombasa', '1075', 'May', '2', 'Apr-May']
['Machakos', '811', 'Nov', '6', 'Nov-Dec']


## Planting Advisory

Four of the five counties (Nairobi, Nakuru, Kisumu and Mombasa) receive their heaviest two-month rainfall in April–May, so farmers there should prepare land in March and plant as the April rains begin. Machakos is the exception: its best window is November–December (278 mm), so farmers there should plan around the short rains rather than the March–May season. Kisumu is the wettest county (1,371 mm a year, no dry months), while Machakos is the driest (811 mm, six dry months), so Machakos farmers should favour drought-tolerant crops and water harvesting. Nakuru has the flattest pattern, with only 145 mm in April, so staggered planting will reduce risk there. **Caveat:** two Machakos readings (April and September) were missing and replaced with an estimate of 68 mm, so Machakos totals and its dry-month count are less certain and the advice should be confirmed against local forecasts before each SMS is sent.

## Data Quality Notes

- **Source of errors:** The Machakos row was typed in by hand and arrived as text. Two entries were invalid: April (`"N/A"`) and September (an empty string).
- **Cleaning method:** Invalid entries were caught with a specific `except ValueError` and replaced by the rounded mean of the 10 valid Machakos months (67.5 → 68 mm). Both imputed months are printed in Section 2.
- **Whitespace:** `" 20 "` (June) was valid and converted to 20 because `int()` ignores surrounding spaces.
- **Effect on results:** The imputed values change the Machakos annual total (811 mm) and could change its dry-month count if the true readings were below 50 mm. Machakos's wettest month (November) and its planting window (November–December) do not depend on the imputed months.
- **Limitation:** The dataset holds a single year of monthly totals per county, so it cannot show year-to-year variation. The other four counties arrived clean and were not altered.